# Monocular / disparity / distance comparison

Three-experiment per-cell comparison: builds the `distance`, `goggles-monocular`, and `goggles-disparity` sessions, computes reliability across all three with `CrossExperimentAnalysis`, and plots each reliable cell's stacked response with `CrossExperimentVisualization.make_combined_plot`.

In [ ]:
import sys, os
import matplotlib.pyplot as plt

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\radial-analysis", base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.RadialDevice import RadialDevice
from src.data.MonocularDevice import MonocularDevice
from src.data.DisparityDevice import DisparityDevice
from src.comparison.CrossExperimentAnalysis import CrossExperimentAnalysis
from src.comparison.CrossExperimentVisualization import CrossExperimentVisualization

In [ ]:
mouse, date = "SAM045", "260715"
R = RadialDevice(mouse_name=mouse, record_date=date, experiment_name="distance")
M = MonocularDevice(mouse_name=mouse, record_date=date, experiment_name="goggles-monocular")
D = DisparityDevice(mouse_name=mouse, record_date=date, experiment_name="goggles-disparity")

In [ ]:
# reliability across all three experiments
cea = CrossExperimentAnalysis.from_devices(R, M, D)
cea.print_summary()

# split-half rho floor -- either one scalar for all three, or a dict to tune
# each experiment separately (a missing key means no threshold for that one)
RHO = {'distance': 0.8, 'monocular': 0.7, 'disparity': 0.75}
sets = cea.reliable_each_thresh(RHO, monocular='any')
all_three = sorted(set(sets['distance']) & set(sets['monocular']) & set(sets['disparity']))
print(f"{len(all_three)} cells reliable in all three experiments at rho >= {RHO}")

In [ ]:
# per-cell response plots, stacked: monocular | disparity | distance
viz = CrossExperimentVisualization(R, M, D)
for cell in all_three:
    viz.make_combined_plot(int(cell))
    plt.show()